In [1]:
import sys
if "code" not in sys.path: sys.path.insert(0, "code")   # model modules live in code/
from tile_model import Hardware, DMA, VCPU
from fa import flash_attention, _array_matmul, _balanced_split

# Causal prefill, one Q head, one query tile x one key block (T = S = Br = Bc).
hw = Hardware()
T = S = Br = Bc = 256 #sequence length Q = sequence length KV
H, R = 128, 128 # attention head dim
m, c, diag = Br, Bc, 0          # causal block touching the diagonal at key 0
dma = DMA(hw=hw)

# GEMM plans (no hardware time): SA1/SA2 rows, jobs, packed/raw bytes.
t_qk = _array_matmul(m, c, H, _balanced_split(m, c, H, R, hw, diag, 'cols'), R, hw, diag, 'cols')
t_pv = _array_matmul(m, H, c, _balanced_split(m, H, c, R, hw, diag, 'depth'), R, hw, diag, 'depth')


ops = [
    ('Load K16',                  dma.transfer((S, H), 16, 'read')),
    ('Quantize K',                VCPU(4*S*H, S*H + 4, hw).launch_cycle()), # 2 bytes * 2 reads, SH write int8 + FP32 scale = 4.  2reads = read amax + map amax to int8
    ('Load V16',                  dma.transfer((S, H), 16, 'read')),
    ('Quantize V + transpose',    VCPU(4*S*H, S*H + 4, hw).launch_cycle()),
    ('load Q16',                  dma.transfer((m, H), 16, 'read')),
    ('quantize Q + init U,l,m',   VCPU(4*m*H, 5*m*H + 8*m + 4, hw).launch_cycle()), # Q8, quantized queries + U FP32, 8 m =  l m FP32, 4 = Qscale
    ('QK pack',                   VCPU((m + c)*H, t_qk['packed_bytes'], hw).launch_cycle()),
    ('QK  (SA1+SA2 GEMM)',        max(t_qk['end'])),
    ('QK reconstruct',            VCPU(t_qk['raw_bytes'] + 8, 4*m*c, hw).launch_cycle()),

    # FP32 scores Sread twice 2 * 4 *mc m,l = 4*m + 4*m
    # write P8 m*c + m alpha l = 3 *4m
    ('softmax + P8',              VCPU(8*m*c + 8*m, m*c + 12*m, hw).launch_cycle()),
    ('PV pack',                   VCPU((m + H)*c, t_pv['packed_bytes'], hw).launch_cycle()),
    ('PV  (SA1+SA2 GEMM)',        max(t_pv['end'])),

    # t_pv['raw_bytes']= INT16 GEMM out,  4 V FP32 scale, 4mh = out C = PV fp32
    ('PV reconstruct',            VCPU(t_pv['raw_bytes'] + 4, 4*m*H, hw).launch_cycle()),
    ('update U,l,m',              VCPU(8*m*H + 12*m, 4*m*H + 8*m, hw).launch_cycle()),
    ('normalize + encode O16',    VCPU(4*m*H + 4*m, 2*m*H, hw).launch_cycle()),
    ('store O16',                 dma.transfer((m, H), 16, 'write')),
]

ref = flash_attention(T=T, S=S, H=H, G=1, Br=Br, Bc=Bc, R=R, hw=hw, overlap=False, verbose=False)
for (name, cyc), step in zip(ops, ref['steps']):
    print(f'{name:28s} {cyc:9,d} cycles   fa: {step["cycles"]:9,d}  {step["name"]}')
total = sum(cyc for _, cyc in ops)
print(f'\nTotal {total:,} cycles ({hw.time_us(total):.3f} us); fa serial {ref["cycles"]:,}')
assert total == ref['cycles'] and len(ops) == len(ref['steps'])

Load K16                         1,424 cycles   fa:     1,424  Load K16
Quantize K                       1,581 cycles   fa:     1,581  Quantize K
Load V16                         1,424 cycles   fa:     1,424  Load V16
Quantize V + transpose           1,581 cycles   fa:     1,581  Quantize V + transpose
load Q16                         1,424 cycles   fa:     1,424  h0 q0 load Q16
quantize Q + init U,l,m          2,621 cycles   fa:     2,621  h0 q0 quantize Q + init U,l,m
QK pack                          1,580 cycles   fa:     1,580  h0 q0 k0 QK pack
QK  (SA1+SA2 GEMM)              12,084 cycles   fa:    12,084  h0 q0 k0 QK
QK reconstruct                   2,909 cycles   fa:     2,909  h0 q0 k0 QK reconstruct
softmax + P8                     4,948 cycles   fa:     4,948  h0 q0 k0 softmax + P8
PV pack                          2,092 cycles   fa:     2,092  h0 q0 k0 PV pack
PV  (SA1+SA2 GEMM)              14,820 cycles   fa:    14,820  h0 q0 k0 PV
PV reconstruct                   2,093 cycl